# Dimension Row Estimation — STATS19 Data Warehouse (12 Dimensions)

**Mục tiêu**: Profiling ước tính số dòng dự kiến cho **toàn bộ 12 bảng Dimension** trong schema Snowflake.

**Nguồn dữ liệu**: Đọc từ file **Clean** (`Processed/`). KHÔNG đọc file Raw.

## Phương pháp
- **Distinct rows** = Số tổ hợp DISTINCT của các cột grain có trong dữ liệu thực tế.
- **Unknown row** = Kiểm tra trực tiếp trong data: nếu có bất kỳ tổ hợp nào chứa giá trị `'Unknown'` hoặc `NaN` thì gán bằng 1 (vì khi load vào bảng Dim chỉ dùng 1 dòng placeholder Unknown duy nhất), ngược lại bằng 0.
- **Expected rows** = Distinct rows + Unknown row.

**Công thức tổng quát**:
```
Expected Rows = |DISTINCT(grain_columns)| + (1 nếu data có chứa Unknown/NaN ngược lại 0)
```

## Danh sách 12 Dimension

| # | Dimension | Loại | Nguồn file | Grain |
|---|---|---|---|---|
| 1 | DimDate | Trực tiếp | collisions_clean.csv | collision_date |
| 2 | DimTimeBand | Phụ (Snowflake) | collisions_clean.csv | time_band (derived từ hour) |
| 3 | DimTime | Trực tiếp | collisions_clean.csv | collision_time (HH:MM) |
| 4 | DimLocalAuthority | Phụ (Snowflake) | collisions_clean.csv | local_authority_district_code |
| 5 | DimLocation | Trực tiếp | collisions_clean.csv | longitude, latitude, local_authority_district_code |
| 6 | DimRoad | Trực tiếp | collisions_clean.csv | first_road_class, road_type, speed_limit_mph, area_type |
| 7 | DimEnvironment | Trực tiếp (Junk) | collisions_clean.csv | light_conditions, weather_conditions, road_surface_conditions |
| 8 | DimJunction | Trực tiếp | collisions_clean.csv | junction_detail, junction_control, pedestrian_crossing |
| 9 | DimSeverity | Trực tiếp (Role-playing) | collisions + casualties | UNION(collision_severity, casualty_severity) |
| 10 | DimPedestrianContext | Phụ (Snowflake) | casualties_clean.csv | pedestrian_location, pedestrian_movement |
| 11 | DimCasualtyProfile | Trực tiếp | casualties_clean.csv | casualty_age_band, casualty_sex, casualty_class, casualty_type, pedestrian_location, pedestrian_movement |
| 12 | DimVehicleProfile | Trực tiếp | vehicles_clean.csv | vehicle_type, driver_age_band, driver_sex, vehicle_manoeuvre, skidding_and_overturning, vehicle_leaving_carriageway |


In [3]:
# =============================================================
# CELL 1 — Import và cấu hình đường dẫn
# =============================================================
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

BASE_DIR        = Path(r'D:/OLAP/Data/Processed')
COLLISIONS_FILE = BASE_DIR / 'collisions_clean.csv'
VEHICLES_FILE   = BASE_DIR / 'vehicles_clean.csv'
CASUALTIES_FILE = BASE_DIR / 'casualties_clean.csv'

UNKNOWN_LABELS = {'Unknown', 'Not known', 'Not reported', 'Data missing or out of range'}

print('Kiểm tra file:')
for f in [COLLISIONS_FILE, VEHICLES_FILE, CASUALTIES_FILE]:
    status = 'OK' if f.exists() else 'NOT FOUND'
    mb = f.stat().st_size / 1_048_576 if f.exists() else 0
    print(f'  {f.name}: {status} ({mb:.1f} MB)')


Kiểm tra file:
  collisions_clean.csv: OK (112.4 MB)
  vehicles_clean.csv: OK (74.0 MB)
  casualties_clean.csv: OK (70.3 MB)


In [4]:
# =============================================================
# CELL 2 — Load dữ liệu từ file Clean
# =============================================================

COLLISIONS_COLS = [
    'collision_id', 'collision_year', 'collision_date', 'collision_time', 'longitude', 'latitude',
    'local_authority_district_code', 'collision_severity', 'first_road_class', 'road_type',
    'speed_limit_mph', 'junction_detail', 'junction_control', 'pedestrian_crossing',
    'light_conditions', 'weather_conditions', 'road_surface_conditions', 'area_type'
]

VEHICLES_COLS = [
    'collision_id', 'vehicle_reference', 'vehicle_type', 'vehicle_manoeuvre', 'driver_age_band',
    'driver_sex', 'skidding_and_overturning', 'vehicle_leaving_carriageway'
]

CASUALTIES_COLS = [
    'collision_id', 'casualty_reference', 'vehicle_reference', 'casualty_class', 'casualty_sex',
    'casualty_age', 'casualty_age_band', 'casualty_severity', 'casualty_type',
    'pedestrian_location', 'pedestrian_movement'
]

print('Đọc collisions_clean.csv...')
df_col = pd.read_csv(COLLISIONS_FILE, usecols=COLLISIONS_COLS, dtype=str, low_memory=False)
print(f'  {len(df_col):,} dòng | {len(df_col.columns)} cột')

print('Đọc vehicles_clean.csv...')
df_veh = pd.read_csv(VEHICLES_FILE, usecols=VEHICLES_COLS, dtype=str, low_memory=False)
print(f'  {len(df_veh):,} dòng | {len(df_veh.columns)} cột')

print('Đọc casualties_clean.csv...')
df_cas = pd.read_csv(CASUALTIES_FILE, usecols=CASUALTIES_COLS, dtype=str, low_memory=False)
print(f'  {len(df_cas):,} dòng | {len(df_cas.columns)} cột')
print('Done.')


Đọc collisions_clean.csv...
  513,801 dòng | 18 cột
Đọc vehicles_clean.csv...
  937,265 dòng | 8 cột
Đọc casualties_clean.csv...
  652,821 dòng | 11 cột
Done.


In [5]:
# =============================================================
# CELL 3 — Hàm profiling
# =============================================================

def count_unknown_in_grain(df, grain_cols):
    """Đếm số dòng DISTINCT có ít nhất 1 cột chứa Unknown hoặc NaN."""
    subset = df[grain_cols].drop_duplicates()
    mask = subset.isna().any(axis=1)
    for col in grain_cols:
        mask = mask | subset[col].isin(UNKNOWN_LABELS)
    return int(mask.sum())


def profile_dimension(dim_name, df, grain_cols, source_file, note=''):
    """
    Profiling một Dimension theo grain thực tế.

    Công thức:
        distinct_rows = |DISTINCT(grain_cols)|
        unknown_row   = 1 (nếu có data Unknown/NaN) else 0
        expected_rows = distinct_rows + unknown_row
    """
    distinct_df     = df[grain_cols].drop_duplicates()
    distinct_rows   = len(distinct_df)
    unknown_count   = count_unknown_in_grain(df, grain_cols)
    
    unknown_row     = 1 if unknown_count > 0 else 0
    expected_rows   = distinct_rows + unknown_row
    
    return {
        'Dimension'        : dim_name,
        'Source File'      : source_file,
        'Grain Columns'    : ', '.join(grain_cols),
        'Total Source Rows': len(df),
        'Distinct rows'    : distinct_rows,
        'Unknown in data'  : unknown_count,
        'Unknown row'      : unknown_row,  # 1 hoac 0
        'Expected rows'    : expected_rows,
        'Note'             : note,
    }


all_results = []   # Tập hợp kết quả tất cả dimension
print('Hàm profiling đã sẵn sàng.')


Hàm profiling đã sẵn sàng.


---
## 1. DimDate

**Grain**: `collision_date` (distinct ngày có trong dữ liệu)  
**Nguồn**: `collisions_clean.csv`

**Công thức**:
```
distinct_rows = |DISTINCT(collision_date)|
expected_rows = distinct_rows + unknown_row
```


In [6]:
# =============================================================
# CELL 4 — DimDate
# =============================================================

result = profile_dimension(
    'DimDate', df_col, ['collision_date'], 'collisions_clean.csv',
    note='Script đếm số ngày thực tế có trong data.'
)
all_results.append(result)

print('=== DimDate ===')
for k, v in result.items():
    print(f'  {k:<25}: {v}')


=== DimDate ===
  Dimension                : DimDate
  Source File              : collisions_clean.csv
  Grain Columns            : collision_date
  Total Source Rows        : 513801
  Distinct rows            : 1826
  Unknown in data          : 0
  Unknown row              : 0
  Expected rows            : 1826
  Note                     : Script đếm số ngày thực tế có trong data.


---
## 2. DimTimeBand (Sub-dimension)

**Grain**: Time band được định nghĩa theo khung giờ (Hour interval)  
**Nguồn**: `collisions_clean.csv` — derive từ cột `collision_time`


In [7]:
# =============================================================
# CELL 5 — DimTimeBand
# =============================================================

def assign_time_band(time_str):
    """Map HH:MM string sang time band (7 bands)."""
    try:
        hour = int(str(time_str).split(':')[0])
        if 0 <= hour <= 3:   return 'Late Night'
        if 4 <= hour <= 6:   return 'Early Morning'
        if 7 <= hour <= 9:   return 'Morning Peak'
        if 10 <= hour <= 12: return 'Mid-day'
        if 13 <= hour <= 15: return 'Afternoon'
        if 16 <= hour <= 18: return 'Evening Peak'
        if 19 <= hour <= 23: return 'Evening'
    except (ValueError, AttributeError):
        pass
    return 'Unknown'

df_tb = df_col[['collision_time']].copy()
df_tb['time_band'] = df_tb['collision_time'].apply(assign_time_band)

distinct_bands = df_tb['time_band'].nunique()
unknown_count  = int((df_tb['time_band'] == 'Unknown').any())
unknown_row    = 1 if unknown_count > 0 else 0
expected_rows  = distinct_bands + unknown_row

result = {
    'Dimension'        : 'DimTimeBand',
    'Source File'      : 'collisions_clean.csv',
    'Grain Columns'    : 'time_band (derived: hour interval cua collision_time)',
    'Total Source Rows': len(df_col),
    'Distinct rows'    : distinct_bands,
    'Unknown in data'  : unknown_count,
    'Unknown row'      : unknown_row,
    'Expected rows'    : expected_rows,
    'Note'             : '7 bands thực tế trong data',
}
all_results.append(result)

print('=== DimTimeBand ===')
for k, v in result.items():
    print(f'  {k:<25}: {v}')


=== DimTimeBand ===
  Dimension                : DimTimeBand
  Source File              : collisions_clean.csv
  Grain Columns            : time_band (derived: hour interval cua collision_time)
  Total Source Rows        : 513801
  Distinct rows            : 7
  Unknown in data          : 0
  Unknown row              : 0
  Expected rows            : 7
  Note                     : 7 bands thực tế trong data


---
## 3. DimTime

**Grain**: `collision_time` (distinct HH:MM có trong dữ liệu)  
**Nguồn**: `collisions_clean.csv`


In [8]:
# =============================================================
# CELL 6 — DimTime
# =============================================================

result = profile_dimension(
    'DimTime', df_col, ['collision_time'], 'collisions_clean.csv'
)
all_results.append(result)

print('=== DimTime ===')
for k, v in result.items():
    print(f'  {k:<25}: {v}')


=== DimTime ===
  Dimension                : DimTime
  Source File              : collisions_clean.csv
  Grain Columns            : collision_time
  Total Source Rows        : 513801
  Distinct rows            : 1440
  Unknown in data          : 0
  Unknown row              : 0
  Expected rows            : 1440
  Note                     : 


---
## 4. DimLocalAuthority (Sub-dimension)

**Grain**: `local_authority_district_code` (mã quận/huyện)  
**Nguồn**: `collisions_clean.csv`


In [9]:
# =============================================================
# CELL 7 — DimLocalAuthority
# =============================================================

result = profile_dimension(
    'DimLocalAuthority', df_col, ['local_authority_district_code'], 'collisions_clean.csv'
)
all_results.append(result)

print('=== DimLocalAuthority ===')
for k, v in result.items():
    print(f'  {k:<25}: {v}')


=== DimLocalAuthority ===
  Dimension                : DimLocalAuthority
  Source File              : collisions_clean.csv
  Grain Columns            : local_authority_district_code
  Total Source Rows        : 513801
  Distinct rows            : 370
  Unknown in data          : 0
  Unknown row              : 0
  Expected rows            : 370
  Note                     : 


---
## 5. DimLocation

**Grain**: `longitude` × `latitude` × `local_authority_district_code`  
**Nguồn**: `collisions_clean.csv`


In [10]:
# =============================================================
# CELL 8 — DimLocation
# =============================================================

LOC_GRAIN = ['longitude', 'latitude', 'local_authority_district_code']
result = profile_dimension('DimLocation', df_col, LOC_GRAIN, 'collisions_clean.csv')
all_results.append(result)

print('=== DimLocation ===')
for k, v in result.items():
    print(f'  {k:<25}: {v}')


=== DimLocation ===
  Dimension                : DimLocation
  Source File              : collisions_clean.csv
  Grain Columns            : longitude, latitude, local_authority_district_code
  Total Source Rows        : 513801
  Distinct rows            : 512359
  Unknown in data          : 33
  Unknown row              : 1
  Expected rows            : 512360
  Note                     : 


---
## 6. DimRoad

**Grain**: `first_road_class` × `road_type` × `speed_limit_mph` × `area_type`  
**Nguồn**: `collisions_clean.csv`


In [11]:
# =============================================================
# CELL 9 — DimRoad
# =============================================================

ROAD_GRAIN = ['first_road_class', 'road_type', 'speed_limit_mph', 'area_type']
result = profile_dimension('DimRoad', df_col, ROAD_GRAIN, 'collisions_clean.csv')
all_results.append(result)

print('=== DimRoad ===')
for k, v in result.items():
    print(f'  {k:<25}: {v}')


=== DimRoad ===
  Dimension                : DimRoad
  Source File              : collisions_clean.csv
  Grain Columns            : first_road_class, road_type, speed_limit_mph, area_type
  Total Source Rows        : 513801
  Distinct rows            : 389
  Unknown in data          : 66
  Unknown row              : 1
  Expected rows            : 390
  Note                     : 


---
## 7. DimEnvironment (Junk Dimension)

**Grain**: `light_conditions` × `weather_conditions` × `road_surface_conditions`  
**Nguồn**: `collisions_clean.csv`


In [12]:
# =============================================================
# CELL 10 — DimEnvironment
# =============================================================

ENV_GRAIN = ['light_conditions', 'weather_conditions', 'road_surface_conditions']
result = profile_dimension('DimEnvironment', df_col, ENV_GRAIN, 'collisions_clean.csv')
all_results.append(result)

print('=== DimEnvironment ===')
for k, v in result.items():
    print(f'  {k:<25}: {v}')


=== DimEnvironment ===
  Dimension                : DimEnvironment
  Source File              : collisions_clean.csv
  Grain Columns            : light_conditions, weather_conditions, road_surface_conditions
  Total Source Rows        : 513801
  Distinct rows            : 228
  Unknown in data          : 88
  Unknown row              : 1
  Expected rows            : 229
  Note                     : 


---
## 8. DimJunction

**Grain**: `junction_detail` × `junction_control` × `pedestrian_crossing`  
**Nguồn**: `collisions_clean.csv`


In [13]:
# =============================================================
# CELL 11 — DimJunction
# =============================================================

JCT_GRAIN = ['junction_detail', 'junction_control', 'pedestrian_crossing']
result = profile_dimension('DimJunction', df_col, JCT_GRAIN, 'collisions_clean.csv')
all_results.append(result)

print('=== DimJunction ===')
for k, v in result.items():
    print(f'  {k:<25}: {v}')


=== DimJunction ===
  Dimension                : DimJunction
  Source File              : collisions_clean.csv
  Grain Columns            : junction_detail, junction_control, pedestrian_crossing
  Total Source Rows        : 513801
  Distinct rows            : 295
  Unknown in data          : 116
  Unknown row              : 1
  Expected rows            : 296
  Note                     : 


---
## 9. DimSeverity (Role-playing Dimension)

**Grain**: UNION distinct values từ `collision_severity` và `casualty_severity`
**Nguồn**: `collisions_clean.csv` + `casualties_clean.csv`


In [14]:
# =============================================================
# CELL 12 — DimSeverity (Role-playing)
# =============================================================

sev_collision = set(df_col['collision_severity'].dropna().unique())
sev_casualty  = set(df_cas['casualty_severity'].dropna().unique())
sev_union     = sev_collision | sev_casualty

unknown_col = sev_collision & UNKNOWN_LABELS
unknown_cas = sev_casualty  & UNKNOWN_LABELS

distinct_rows = len(sev_union)
unknown_count = len(unknown_col | unknown_cas)
unknown_row   = 1 if unknown_count > 0 else 0
expected_rows = distinct_rows + unknown_row

result = {
    'Dimension'        : 'DimSeverity',
    'Source File'      : 'collisions_clean.csv + casualties_clean.csv',
    'Grain Columns'    : 'UNION(collision_severity, casualty_severity)',
    'Total Source Rows': len(df_col) + len(df_cas),
    'Distinct rows'    : distinct_rows,
    'Unknown in data'  : unknown_count,
    'Unknown row'      : unknown_row,
    'Expected rows'    : expected_rows,
    'Note'             : 'Role-playing dim',
}
all_results.append(result)

print('=== DimSeverity ===')
for k, v in result.items():
    print(f'  {k:<25}: {v}')


=== DimSeverity ===
  Dimension                : DimSeverity
  Source File              : collisions_clean.csv + casualties_clean.csv
  Grain Columns            : UNION(collision_severity, casualty_severity)
  Total Source Rows        : 1166622
  Distinct rows            : 3
  Unknown in data          : 0
  Unknown row              : 0
  Expected rows            : 3
  Note                     : Role-playing dim


---
## 10. DimPedestrianContext (Sub-dimension)

**Grain**: `pedestrian_location` × `pedestrian_movement`  
**Nguồn**: `casualties_clean.csv`


In [15]:
# =============================================================
# CELL 13 — DimPedestrianContext
# =============================================================

PED_GRAIN = ['pedestrian_location', 'pedestrian_movement']
result = profile_dimension('DimPedestrianContext', df_cas, PED_GRAIN, 'casualties_clean.csv')
all_results.append(result)

print('=== DimPedestrianContext ===')
for k, v in result.items():
    print(f'  {k:<25}: {v}')


=== DimPedestrianContext ===
  Dimension                : DimPedestrianContext
  Source File              : casualties_clean.csv
  Grain Columns            : pedestrian_location, pedestrian_movement
  Total Source Rows        : 652821
  Distinct rows            : 89
  Unknown in data          : 19
  Unknown row              : 1
  Expected rows            : 90
  Note                     : 


---
## 11. DimCasualtyProfile

**Grain**: `casualty_age_band` × `casualty_sex` × `casualty_class` × `casualty_type` × `pedestrian_location` × `pedestrian_movement`  
**Nguồn**: `casualties_clean.csv`


In [16]:
# =============================================================
# CELL 14 — DimCasualtyProfile
# =============================================================

CAS_GRAIN = [
    'casualty_age_band', 'casualty_sex', 'casualty_class', 'casualty_type',
    'pedestrian_location', 'pedestrian_movement'
]
result = profile_dimension('DimCasualtyProfile', df_cas, CAS_GRAIN, 'casualties_clean.csv')
all_results.append(result)

print('=== DimCasualtyProfile ===')
for k, v in result.items():
    print(f'  {k:<25}: {v}')


=== DimCasualtyProfile ===
  Dimension                : DimCasualtyProfile
  Source File              : casualties_clean.csv
  Grain Columns            : casualty_age_band, casualty_sex, casualty_class, casualty_type, pedestrian_location, pedestrian_movement
  Total Source Rows        : 652821
  Distinct rows            : 2477
  Unknown in data          : 826
  Unknown row              : 1
  Expected rows            : 2478
  Note                     : 


---
## 12. DimVehicleProfile

**Grain**: `vehicle_type` × `driver_age_band` × `driver_sex` × `vehicle_manoeuvre` × `skidding_and_overturning` × `vehicle_leaving_carriageway`  
**Nguồn**: `vehicles_clean.csv`


In [17]:
# =============================================================
# CELL 15 — DimVehicleProfile
# =============================================================

VEH_GRAIN = [
    'vehicle_type', 'driver_age_band', 'driver_sex',
    'vehicle_manoeuvre', 'skidding_and_overturning', 'vehicle_leaving_carriageway'
]
result = profile_dimension('DimVehicleProfile', df_veh, VEH_GRAIN, 'vehicles_clean.csv')
all_results.append(result)

print('=== DimVehicleProfile ===')
for k, v in result.items():
    print(f'  {k:<25}: {v}')


=== DimVehicleProfile ===
  Dimension                : DimVehicleProfile
  Source File              : vehicles_clean.csv
  Grain Columns            : vehicle_type, driver_age_band, driver_sex, vehicle_manoeuvre, skidding_and_overturning, vehicle_leaving_carriageway
  Total Source Rows        : 937265
  Distinct rows            : 20578
  Unknown in data          : 13335
  Unknown row              : 1
  Expected rows            : 20579
  Note                     : 


---
## Tổng hợp Báo cáo — 12 Dimensions

**Công thức**: `Expected Rows = |DISTINCT(grain_cols)| + (1 nếu data có Unknown else 0)`


In [18]:
# =============================================================
# CELL 16 — Báo cáo tổng hợp 12 Dimensions
# =============================================================

report_df = pd.DataFrame([{
    'Dimension'      : r['Dimension'],
    'Source File'    : r['Source File'],
    'Grain'          : r['Grain Columns'],
    'Distinct rows'  : r['Distinct rows'],
    'Unknown in data': r['Unknown in data'],
    'Unknown row'    : r['Unknown row'],
    'Expected rows'  : r['Expected rows'],
    'Note'           : r.get('Note', ''),
} for r in all_results])

print('=' * 80)
print('BÁO CÁO DIMENSION ROW ESTIMATION — ACTUAL ROWS + PLACEHOLDER LOGIC')
print('Công thức: Expected Rows = Distinct rows + 1 (Nếu data chứa Unknown)')
print('=' * 80)
display(report_df)

print('\n--- Bảng tóm tắt (theo format yêu cầu) ---')
summary = report_df[['Dimension', 'Distinct rows', 'Unknown row', 'Expected rows']]
print(summary.to_string(index=False))


BÁO CÁO DIMENSION ROW ESTIMATION — ACTUAL ROWS + PLACEHOLDER LOGIC
Công thức: Expected Rows = Distinct rows + 1 (Nếu data chứa Unknown)


,Dimension,Source File,Grain,Distinct rows,Unknown in data,Unknown row,Expected rows,Note
0,DimDate,collisions_clean.csv,collision_date,1826,0,0,1826,Script đếm số ngày thực tế có trong data.
1,DimTimeBand,collisions_clean.csv,time_band (derived: hour interval cua collisio...,7,0,0,7,7 bands thực tế trong data
2,DimTime,collisions_clean.csv,collision_time,1440,0,0,1440,
3,DimLocalAuthority,collisions_clean.csv,local_authority_district_code,370,0,0,370,
4,DimLocation,collisions_clean.csv,"longitude, latitude, local_authority_district_...",512359,33,1,512360,
5,DimRoad,collisions_clean.csv,"first_road_class, road_type, speed_limit_mph, ...",389,66,1,390,
6,DimEnvironment,collisions_clean.csv,"light_conditions, weather_conditions, road_sur...",228,88,1,229,
7,DimJunction,collisions_clean.csv,"junction_detail, junction_control, pedestrian_...",295,116,1,296,
8,DimSeverity,collisions_clean.csv + casualties_clean.csv,"UNION(collision_severity, casualty_severity)",3,0,0,3,Role-playing dim
9,DimPedestrianContext,casualties_clean.csv,"pedestrian_location, pedestrian_movement",89,19,1,90,



--- Bảng tóm tắt (theo format yêu cầu) ---
           Dimension  Distinct rows  Unknown row  Expected rows
             DimDate           1826            0           1826
         DimTimeBand              7            0              7
             DimTime           1440            0           1440
   DimLocalAuthority            370            0            370
         DimLocation         512359            1         512360
             DimRoad            389            1            390
      DimEnvironment            228            1            229
         DimJunction            295            1            296
         DimSeverity              3            0              3
DimPedestrianContext             89            1             90
  DimCasualtyProfile           2477            1           2478
   DimVehicleProfile          20578            1          20579


In [19]:
# =============================================================
# CELL 17 — Xuất báo cáo ra Excel (.xlsx)
# =============================================================

OUTPUT_DIR  = Path(r'D:/OLAP/Data/Profiling')
OUTPUT_FILE = OUTPUT_DIR / 'dimension_row_estimation_report.xlsx'

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:

    # --- Sheet 1: Summary (đúng format yêu cầu) ---
    summary_df = report_df[['Dimension', 'Distinct rows', 'Unknown row', 'Expected rows']].copy()
    summary_df.to_excel(writer, sheet_name='Summary', index=False)

    # --- Sheet 2: Full Detail (thêm Grain, Source, Note) ---
    report_df.to_excel(writer, sheet_name='Full Detail', index=False)

    # Căn chỉnh độ rộng cột — Sheet Summary
    ws = writer.sheets['Summary']
    ws.column_dimensions['A'].width = 24
    ws.column_dimensions['B'].width = 16
    ws.column_dimensions['C'].width = 14
    ws.column_dimensions['D'].width = 16

    # Căn chỉnh độ rộng cột — Sheet Full Detail
    ws2 = writer.sheets['Full Detail']
    ws2.column_dimensions['A'].width = 24
    ws2.column_dimensions['B'].width = 40
    ws2.column_dimensions['C'].width = 80
    ws2.column_dimensions['D'].width = 20
    ws2.column_dimensions['E'].width = 16
    ws2.column_dimensions['F'].width = 20
    ws2.column_dimensions['G'].width = 14
    ws2.column_dimensions['H'].width = 16
    ws2.column_dimensions['I'].width = 50

print(f'Đã xuất báo cáo: {OUTPUT_FILE}')
print(f'Số dimensions: {len(report_df)}')
print('Sheets: Summary | Full Detail')


Đã xuất báo cáo: D:\OLAP\Data\Profiling\dimension_row_estimation_report.xlsx
Số dimensions: 12
Sheets: Summary | Full Detail
